In [1]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

import jax
import jax.numpy as jnp
from waymax import config, dataloader, dynamics, env, datatypes, visualization
import matplotlib.pyplot as plt

print(jax.devices())

I0000 00:00:1774465166.962699   58923 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[CudaDevice(id=0)]


In [2]:
data_config = config.DatasetConfig(
    path='gs://waymo_open_dataset_motion_v_1_2_0/uncompressed/tf_example/training/training_tfexample.tfrecord-00000-of-01000',
    max_num_objects=16,
)

scenarios = dataloader.simulator_state_generator(data_config)
scenario = next(scenarios)

print(type(scenario))
print(scenario.shape)
print(f"Num objects: {scenario.num_objects}")
print(f"Num timesteps: {scenario.sim_trajectory.num_timesteps}")

I0000 00:00:1774465201.740654   59317 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


<class 'waymax.datatypes.simulator_state.SimulatorState'>
()
Num objects: 16
Num timesteps: 91


In [ ]:
# Cell: what is the SDC? -- self-driving car --> our agent
print(scenario.object_metadata.is_sdc)
print(scenario.object_metadata.is_sdc.shape)

[False  True False False False False False False False False False False
 False False False False]
(16,)


## Object types come from ObjectTypeIds:

* 0 = Unset
* 1 = Vehicle
* 2 = Pedestrian
* 3 = Cyclist
* 4 = Other

In [4]:
# Cell: what types of objects are there?
print(scenario.object_metadata.object_types)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [5]:
# Cell: which objects are valid?
print(scenario.object_metadata.is_valid)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]


In [6]:
# Cell: look at the SDC's current position
traj = scenario.sim_trajectory
print(f"x shape: {traj.x.shape}")
print(f"SDC x over time: {traj.x[1]}")  # index 1 is the SDC
print(f"SDC y over time: {traj.y[1]}")

x shape: (16, 91)
SDC x over time: [350.06186   0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.
   0.        0.        0.        0.        0.        0.        0.     ]
SDC y over time: [150.71342   0.        0.        0.        0.        0.        0.
   0.        0.   

### Trajectories
* log_trajectory = the full recorded ground truth for all 91 timesteps — where the human driver actually went
* sim_trajectory = what the agent has done so far in the simulation 
    * starts with only timestep 0 filled in, gets populated as you call step()

In [7]:
# Cell: compare sim_trajectory vs log_trajectory
print("sim x:", scenario.sim_trajectory.x[1, :5])   # first 5 timesteps
print("log x:", scenario.log_trajectory.x[1, :5])   # first 5 timesteps

sim x: [350.06186   0.        0.        0.        0.     ]
log x: [350.06186 350.72552 351.38367 352.04102 352.69696]


In [9]:
# Cell: set up the environment
dynamics_model = dynamics.InvertibleBicycleModel()
env_config = config.EnvironmentConfig(max_num_objects=16) # By defaults, max_num_objects=128
waymax_env = env.PlanningAgentEnvironment(dynamics_model, env_config)

# Reset to get initial state
state = waymax_env.reset(scenario)
print(f"timestep: {state.timestep}")
print(f"is_done: {state.is_done}")

timestep: 10
is_done: False


#####  DatasetConfig and EnvironmentConfig must always have matching max_num_objects

## Expert action

In [13]:
from waymax import agents

expert_actor = agents.create_expert_actor(dynamics_model)
actor_state = expert_actor.init(jax.random.PRNGKey(0), state)
action = expert_actor.select_action(params=None, state=state, actor_state=None, rng=None)

print(f"action data shape: {action.action.data.shape}")
print(f"action valid shape: {action.action.valid.shape}")
print(f"SDC action: {action.action.data[1]}")
print(f"SDC valid: {action.action.valid[1]}")

action data shape: (16, 2)
action valid shape: (16, 1)
SDC action: [-4.0557861e-01 -1.9343873e-04]
SDC valid: [ True]


In [19]:
action.action.data.shape

(16, 2)

## Actions
* data[0] = acceleration (m/s²) — negative here means the SDC is decelerating
* data[1] = steering angle (radians) — nearly zero here means going roughly straight

In [15]:
print(f"action.action.data.shape: {action.action.data.shape}")
print(f"action.action.valid.shape: {action.action.valid.shape}")
print(f"action.action.shape: {action.action.shape}")

action.action.data.shape: (16, 2)
action.action.valid.shape: (16, 1)
action.action.shape: (16,)


In [16]:
# # Step the environment with the expert action
# next_state = waymax_env.step(state, action.action)

# print(f"timestep before: {state.timestep}")
# print(f"timestep after: {next_state.timestep}")
# print(f"SDC x before: {state.current_sim_trajectory.x[1]}")
# print(f"SDC x after: {next_state.current_sim_trajectory.x[1]}")

In [ ]:
env_config = config.EnvironmentConfig(max_num_objects=16)
waymax_env_base = env.BaseEnvironment(dynamics_model, env_config)
state_base = waymax_env_base.reset(scenario)

print(f"timestep: {state.timestep}")
print(f"is_done: {state.is_done}")

timestep: 10
is_done: False


## Full episode loop

In [ ]:
dynamics_model = dynamics.InvertibleBicycleModel()
env_config = config.EnvironmentConfig(max_num_objects=16)
waymax_env = env.BaseEnvironment(dynamics_model, env_config)
expert_actor = agents.create_expert_actor(dynamics_model)

state = waymax_env.reset(scenario)

sdc_x = []
sdc_y = []

while not state.is_done:
    action = expert_actor.select_action(params=None, state=state, actor_state=None, rng=None)
    state = waymax_env.step(state, action.action)
    
    # collect SDC position — you fill this part in
    # hint: state.current_sim_trajectory.x[?]

print(f"Collected {len(sdc_x)} timesteps")

IndexError: Too many indices: array is 0-dimensional, but 1 were indexed